In [ ]:
# Cell 1: Imports
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense, RepeatVector, TimeDistributed

print('All imports successful.')


ModuleNotFoundError: No module named 'tensorflow'

In [1]:
# Cell 2: Load & Preprocess Data
DATA_PATH = 'live_data.csv'
TIME_STEPS = 10
RESAMPLE_RULE = '1min'

expected_cols = [
    'timestamp', 'machine', 'cpu_usage', 'gpu_wrk_util',
    'avg_mem', 'max_mem', 'avg_gpu_wrk_mem', 'max_gpu_wrk_mem',
    'read', 'write', 'read_count', 'write_count'
]
features = [
    'cpu_usage', 'gpu_wrk_util', 'avg_mem', 'max_mem',
    'avg_gpu_wrk_mem', 'max_gpu_wrk_mem', 'read', 'write',
    'read_count', 'write_count'
]

df = pd.read_csv(DATA_PATH)
if list(df.columns) != expected_cols:
    if len(df.columns) == len(expected_cols):
        df.columns = expected_cols
    else:
        raise ValueError(f'Expected {len(expected_cols)} columns, found {len(df.columns)}.')

# New generator rows use Unix epoch seconds. Legacy rows used dd-mm-yy strings.
numeric_ts = pd.to_numeric(df['timestamp'], errors='coerce')
epoch_ts = pd.to_datetime(numeric_ts, unit='s', errors='coerce')
legacy_ts = pd.to_datetime(df['timestamp'], format='%d-%m-%y %H:%M:%S', errors='coerce')
df['timestamp'] = epoch_ts.fillna(legacy_ts)

original_count = len(df)
df = df.dropna(subset=['timestamp', 'machine']).copy()

# 1970 bomb defusal: never resample old/corrupt epoch timestamps.
df = df[df['timestamp'].dt.year >= 2025].copy()

for col in features:
    df[col] = pd.to_numeric(df[col], errors='coerce')
df = df.dropna(subset=features).sort_values(['machine', 'timestamp']).copy()

unique_servers = df['machine'].dropna().unique()
print(f'Loaded {original_count} rows, kept {len(df)} clean rows (year >= 2025).')
print(f'Unique servers: {unique_servers}')


NameError: name 'pd' is not defined

In [ ]:
# Cell 3: Build Time-Series Sequences per Server
all_sequences = []
sequence_metadata = []
scalers_by_server = {}

for mac in unique_servers:
    server_data = df[df['machine'] == mac].copy()
    min_time = server_data['timestamp'].min()
    max_time = server_data['timestamp'].max()
    time_gap = max_time - min_time

    print(f'\n--- {mac} ---')
    print(f'  First log : {min_time}')
    print(f'  Last log  : {max_time}')
    print(f'  Time span : {time_gap}')

    if time_gap.days > 2:
        print(f'  WARNING: Massive time gap detected for {mac}. Skipping this server.')
        continue

    server_minute = (
        server_data
        .set_index('timestamp')
        .resample(RESAMPLE_RULE)
        .mean(numeric_only=True)
        .dropna()
    )
    print(f'  Resampled to {len(server_minute)} clean 1-minute rows.')

    if len(server_minute) <= TIME_STEPS:
        print(f'  Not enough data for {mac} to create a {TIME_STEPS}-step window. Skipping.')
        continue

    scaler = MinMaxScaler()
    data_scaled = scaler.fit_transform(server_minute[features].values)
    scalers_by_server[mac] = scaler

    for i in range(len(data_scaled) - TIME_STEPS):
        all_sequences.append(data_scaled[i : i + TIME_STEPS])
        sequence_metadata.append({
            'machine': mac,
            'start': server_minute.index[i],
            'end': server_minute.index[i + TIME_STEPS - 1],
        })

print(f'\nTotal sequences built: {len(all_sequences)}')


In [ ]:
# Cell 4: Train / Test Split
if len(all_sequences) == 0:
    raise ValueError('No sequences were built. Collect more data or verify timestamps are valid.')

X = np.array(all_sequences)
print(f'Dataset shape: {X.shape}')

X_train, X_test, train_meta, test_meta = train_test_split(
    X,
    sequence_metadata,
    test_size=0.2,
    random_state=42,
    shuffle=False,
)
print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')


In [ ]:
# Cell 5: Build LSTM Autoencoder, Train, and Detect Anomalies
n_timesteps = X_train.shape[1]
n_features = X_train.shape[2]

model = Sequential([
    Input(shape=(n_timesteps, n_features)),
    LSTM(16, activation='relu', return_sequences=False),
    RepeatVector(n_timesteps),
    LSTM(16, activation='relu', return_sequences=True),
    TimeDistributed(Dense(n_features)),
])

model.compile(optimizer='adam', loss='mse')
model.summary()

history = model.fit(
    X_train, X_train,
    epochs=20,
    batch_size=32,
    validation_split=0.1,
    shuffle=True,
    verbose=1,
)

# Threshold is learned only from training reconstruction loss.
X_train_pred = model.predict(X_train)
train_mse = np.mean(np.power(X_train - X_train_pred, 2), axis=(1, 2))
threshold = np.percentile(train_mse, 99)

X_test_pred = model.predict(X_test)
test_mse = np.mean(np.power(X_test - X_test_pred, 2), axis=(1, 2))
anomalies = test_mse > threshold

print('\n=== Anomaly Detection Results ===')
print(f'Training Reconstruction Error - Mean : {np.mean(train_mse):.6f}')
print(f'Training Reconstruction Error - P99  : {threshold:.6f}')
print(f'Test Reconstruction Error     - Mean : {np.mean(test_mse):.6f}')
print(f'Anomalies Detected                  : {anomalies.sum()} / {len(test_mse)} sequences')

anomaly_indices = np.where(anomalies)[0]
if len(anomaly_indices) > 0:
    print('\nFlagged sequences:')
    for idx in anomaly_indices[:25]:
        meta = test_meta[idx]
        print(
            f"  {meta['machine']} {meta['start']} -> {meta['end']} | "
            f'MSE = {test_mse[idx]:.6f} [ANOMALY]'
        )
else:
    print('\nNo anomalies detected in the held-out test set.')


In [ ]:
# Cell 6: Sanity Check Against Synthetic Injected Failures
failure_scenarios = {
    'memory_leak_oom': {'avg_mem': 1.4, 'max_mem': 1.6},
    'cpu_runaway_deadlock': {'cpu_usage': 1.6, 'read': 0.0, 'write': 0.0, 'read_count': 0.0, 'write_count': 0.0},
    'gpu_crash_cuda_oom': {'gpu_wrk_util': 0.0, 'avg_gpu_wrk_mem': 1.6, 'max_gpu_wrk_mem': 1.6},
    'network_ddos': {'read': 1.7, 'read_count': 1.7, 'cpu_usage': 1.4, 'avg_mem': 1.3, 'max_mem': 1.4},
    'silent_death': {feature: 0.0 for feature in features},
}

feature_index = {feature: idx for idx, feature in enumerate(features)}
base_windows = X_test if len(X_test) else X_train

print('\n=== Injected Failure Sanity Check ===')
for name, overrides in failure_scenarios.items():
    synthetic = base_windows.copy()
    for feature, value in overrides.items():
        synthetic[:, :, feature_index[feature]] = value

    synthetic_pred = model.predict(synthetic, verbose=0)
    synthetic_mse = np.mean(np.power(synthetic - synthetic_pred, 2), axis=(1, 2))
    detected = synthetic_mse > threshold
    print(f'{name}: {detected.sum()} / {len(detected)} windows flagged')
